# GreenMind AI: Notebook 3, disease classifier and the lab-to-field gap

**Settings:** Accelerator = **GPU T4 x2**, Internet = **On**. No inputs needed: PlantVillage (15 classes, `emmarex/plantdisease`)
is downloaded with `kagglehub`, PlantDoc from its official GitHub repository.
Then **Save & Run All (Commit)**, about 1–1.5 h.

In [ ]:
import os, subprocess, sys
os.chdir("/kaggle/working")
if not os.path.exists("greenmind-ai"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/mubashir20023/greenmind-ai"], check=True)
os.chdir("/kaggle/working/greenmind-ai")
assert os.path.exists("tools/inat_fetch_species.py"), "Push the new tools/ folder to GitHub first (see RUN_EXPERIMENTS.md, Step 0)"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm==1.0.19", "scikit-learn"], check=False)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
import glob, os, subprocess, kagglehub

# PlantVillage (15 classes) from Kaggle, PlantDoc (field photos) from its official GitHub repo
pv_root = kagglehub.dataset_download("emmarex/plantdisease")
if not os.path.exists("/kaggle/working/PlantDoc-Dataset"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/pratikkayal/PlantDoc-Dataset", "/kaggle/working/PlantDoc-Dataset"], check=True)

def find_parent(root, marker):
    hits = [d for d in glob.glob(root + "/**/" + marker, recursive=True) if os.path.isdir(d)]
    return sorted({os.path.dirname(h) for h in hits}, key=len)

pv = find_parent(pv_root, "Tomato_healthy")
pd = find_parent("/kaggle/working/PlantDoc-Dataset", "Tomato leaf late blight")
pd = [p for p in pd if p.rstrip("/").endswith("test")] or pd
print("PlantVillage:", pv); print("PlantDoc:", pd)
PV, PD = pv[0], pd[0]

In [ ]:
!python tools/make_split.py --src "{PV}" --min_per_class 30 --dedupe --out_dir /kaggle/working/pv_split

In [ ]:
!python tools/train_classifier.py --manifest /kaggle/working/pv_split/manifest.csv --arch efficientnet_b0 \
    --norm half --resize_mode square --epochs 20 --batch_size 64 --lr 5e-4 --save_full --workers 4 \
    --out /kaggle/working/disease_effb0
!python tools/train_classifier.py --manifest /kaggle/working/pv_split/manifest.csv --arch resnet18 \
    --norm half --resize_mode square --epochs 20 --batch_size 64 --lr 5e-4 --workers 4 \
    --out /kaggle/working/disease_resnet18

In [ ]:
for arch, d in [("efficientnet_b0", "disease_effb0"), ("resnet18", "disease_resnet18")]:
    !python tools/evaluate.py --ckpt /kaggle/working/{d}/best.pth --arch {arch} --norm half --resize_mode square \
        --classes /kaggle/working/{d}/classes.json --imagefolder "{PD}" --label_map tools/plantdoc_label_map.json \
        --out /kaggle/working/results/e4_plantdoc_{d}

In [ ]:
!rm -f /kaggle/working/*/last.pth
!cat /kaggle/working/pv_split/split_stats.json
!for f in /kaggle/working/disease_*/eval_test/metrics.json /kaggle/working/results/*/metrics.json; do echo $f; head -12 $f; done